In [1]:
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [17]:
DEFAULT_SEASON_TYPE = "REG"
DEFAULT_WEEKS = 1  # 18 regular season weeks

def _extract_table(html: str) -> pd.DataFrame:
    """
    Extract the visible stats table from a Next Gen Stats page or saved HTML.
    Prefers the body table under .el-table__body-wrapper to avoid grabbing the header-only table.
    """
    soup = BeautifulSoup(html, "html.parser")
    table = soup.select_one("div.el-table__body-wrapper table.el-table__body") or soup.find("table")
    if table is None:
        print(f"Didn't find table!")
        return pd.DataFrame()

    # Extract header labels if present so we can override pandas' default column names.
    header_cells = [
        _clean_text(th.get_text(" ", strip=True))
        for th in soup.select("div.el-table__header-wrapper th .cell")
        if _clean_text(th.get_text(" ", strip=True))
    ]
    
    # Manual parse to avoid optional lxml/html5lib dependencies.
    rows: list[list[str]] = []
    for tr in table.select("tbody tr"):
        cells = [_clean_text(td.get_text(" ", strip=True)) for td in tr.find_all("td")]
        if any(cells):
            rows.append(cells)

    if not rows:
        return pd.DataFrame(columns=header_cells)

    max_cols = max(len(r) for r in rows)
    col_names = header_cells if header_cells and len(header_cells) == max_cols else [f"col{i+1}" for i in range(max_cols)]

    # Pad short rows so DataFrame construction does not fail.
    padded_rows = [r + [""] * (max_cols - len(r)) for r in rows]
    df = pd.DataFrame(padded_rows, columns=col_names)

    # Final cleanup
    df.columns = [_clean_text(c) for c in df.columns]

    return df
    
@dataclass
class NextGenRequest:
    stat: str  # rushing, receiving, passing
    season: int
    season_type: str = DEFAULT_SEASON_TYPE
    week: int | None = None

    @property
    def url(self) -> str:
        wk = self.week or 1
        return BASE_URL.format(
            stat=self.stat, season=self.season, season_type=self.season_type, week=wk
        )

BASE_URL = "https://nextgenstats.nfl.com/stats/{stat}/{season}/{season_type}/{week}"
API_URL = "https://nextgenstats.nfl.com/api/players/{stat}"
req = NextGenRequest(stat='rushing', season='2025', season_type='REG', week=DEFAULT_WEEKS)
print(BASE_URL.format(stat=req.stat,season=req.season,season_type=req.season_type, week=DEFAULT_WEEKS))
session = requests.Session()


resp = session.get(
    BASE_URL.format(stat=req.stat,season=req.season,season_type=req.season_type, week=DEFAULT_WEEKS),
    params={"season": req.season, "seasonType": req.season_type, "week": req.week},
    headers={
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "Accept": "application/json",
        "Referer": req.url,
    },
    timeout=30,
)

try:
    resp = session.get(
        req.url,
        headers={
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
            "Accept-Language": "en-US,en;q=0.9",
        },
        timeout=30,
    )
    resp.raise_for_status()
    print(_extract_table(resp.text))
except:
    print("Error Occurred")

https://nextgenstats.nfl.com/stats/rushing/2025/REG/1
Didn't find table!
Empty DataFrame
Columns: []
Index: []


In [20]:
url="/v2/sports/football/leagues/nfl/events?dates=2025"
core_api="https://sports.core.api.espn.com/"
request_url= core_api + url
resp=session.get( 
    request_url,
    headers={
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "Accept": "application/json",
        "Referer": req.url,
    },
    timeout=30,
)

In [22]:
resp.text

'{"count":335,"pageIndex":1,"pageSize":25,"pageCount":14,"items":[{"$ref":"http://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401772809?lang=en&region=us"},{"$ref":"http://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401772915?lang=en&region=us"},{"$ref":"http://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401772954?lang=en&region=us"},{"$ref":"http://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401772807?lang=en&region=us"},{"$ref":"http://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401772916?lang=en&region=us"},{"$ref":"http://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401772808?lang=en&region=us"},{"$ref":"http://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401772952?lang=en&region=us"},{"$ref":"http://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401772950?lang=en&region=us"},{"$ref":"http://sports.core.api.espn.com/v2/sports/football/le